# Repaso: conceptos clave de POO

Este notebook es un resumen rápido de lo que hemos visto en los vídeos de
Programación Orientada a Objetos. No pretende explicar cada cosa en detalle (para eso
están los vídeos), sino refrescar las ideas clave con un ejemplo mínimo de cada una
antes de ponernos a practicar.


---

## 0. ¿Qué es el paradigma de Programación Orientada a Objetos?

Un **paradigma de programación** es una forma de *pensar* y *organizar* el código. No es
una sintaxis ni un lenguaje: es la respuesta a la pregunta *"¿cómo estructuro un programa
para que sea entendible, mantenible y ampliable?"*. Vamos a comparar los dos que nos
interesan:

| Paradigma | ¿Cómo organiza el programa? | Unidad básica | Ejemplo típico |
|---|---|---|---|
| **Secuencial / imperativo** | Una lista de instrucciones que se ejecutan de arriba abajo | La instrucción | Un script que lee un CSV y saca un resumen |
| **Orientado a objetos** | Agrupa **datos + comportamiento** en **objetos** que se comunican entre sí | El objeto (y la clase que lo describe) | Una `CuentaBancaria` con saldo y métodos `ingresar()` / `retirar()` |

### El problema de la programación secuencial

En un programa secuencial, los **datos** son variables sueltas y las **operaciones**
son instrucciones (o funciones) que viven separadas de esos datos:

```python
saldo = 100
titular = "Ana"

def retirar(saldo, cantidad):
    if cantidad > saldo:
        raise ValueError("Saldo insuficiente")
    return saldo - cantidad

saldo = retirar(saldo, 30)
```

Funciona mientras el programa es pequeño. Pero conforme crece:

- Los datos están **sueltos**: nada garantiza que `saldo` y `titular` pertenezcan a la misma cuenta.
- Nadie impide hacer `saldo = -500` desde cualquier punto del programa.
- Si añadimos 20 cuentas, acabamos con listas paralelas o diccionarios que hay que mantener sincronizados a mano.
- Las funciones acaban llenas de `if tipo == "ahorro": ... elif tipo == "corriente": ...`

### La idea central de la POO

La POO propone **juntar en una misma unidad, el objeto, los datos y las operaciones que
tienen sentido sobre esos datos**:

```python
class CuentaBancaria:
    def __init__(self, titular, saldo):
        self.titular = titular
        self.saldo = saldo

    def retirar(self, cantidad):
        if cantidad > self.saldo:
            raise ValueError("Saldo insuficiente")
        self.saldo -= cantidad
```

Ahora `saldo` y `titular` viajan juntos, y la única forma "oficial" de modificar el saldo
es a través de los métodos de la cuenta. El programa deja de ser *una secuencia de
instrucciones sobre datos sueltos* y pasa a ser *un conjunto de objetos que colaboran
enviándose mensajes* (llamando a sus métodos).

### Los cuatro pilares de la POO

| Pilar | Idea | En Python |
|---|---|---|
| **Abstracción** | Modelar solo lo relevante de un concepto real y esconder los detalles | Clases, clases abstractas (ABC) |
| **Encapsulación** | Datos y comportamiento juntos; el estado se modifica a través de métodos | `self.atributo`, métodos, convención `_privado` |
| **Herencia** | Una clase puede *reutilizar y especializar* otra ("es un") | `class Perro(Animal)`, `super()` |
| **Polimorfismo** | Objetos distintos responden al mismo mensaje cada uno a su manera | Mismo nombre de método en varias clases (lo veremos con las ABC) |


---

## 1. Clase, objeto y estado

- Una **clase** es el *molde*: describe qué datos y qué comportamiento tienen las cosas de un tipo.
- Un **objeto** (instancia) es una *pieza concreta* creada con ese molde.
- `__init__` es el **constructor**: se ejecuta al crear la instancia e inicializa su **estado**.
- `self` es *el propio objeto* sobre el que se está trabajando. Python lo pasa automáticamente.
- Los **métodos** son funciones definidas dentro de la clase que operan sobre `self`.

In [1]:
class Usuario:
    def __init__(self, nombre, edad):
        self.nombre = nombre  # atributo de instancia
        self.edad = edad

    def saludar(self):
        return f"Hola, soy {self.nombre} y tengo {self.edad} años"

    def es_mayor_de_edad(self):
        return self.edad >= 18


ana = Usuario("Ana", 25)
luis = Usuario("Luis", 15)

print(ana.saludar())
print(luis.saludar())
print(ana.es_mayor_de_edad(), luis.es_mayor_de_edad())

Hola, soy Ana y tengo 25 años
Hola, soy Luis y tengo 15 años
True False


> 💡 Cada objeto tiene **su propio estado**: `ana.edad` y `luis.edad` son independientes.
> `Usuario` define el molde; `Usuario(...)` fabrica una pieza nueva cada vez.

---

## 2. Atributos de instancia vs atributos de clase

- **Atributo de instancia**: se define en `__init__` con `self.x = ...`. Cada objeto tiene el suyo.
- **Atributo de clase**: se define directamente en el cuerpo de la clase. Es **compartido** por todas las instancias.

In [2]:
class Usuario:
    plataforma = "MiApp"  # atributo de clase: compartido

    def __init__(self, nombre):
        self.nombre = nombre  # atributo de instancia: propio de cada objeto


ana = Usuario("Ana")
luis = Usuario("Luis")

print(ana.plataforma, luis.plataforma)

Usuario.plataforma = "MiApp v2"  # cambia para TODOS
print(ana.plataforma, luis.plataforma)

MiApp MiApp
MiApp v2 MiApp v2


---

## 3. Los tres tipos de métodos

| Tipo | Decorador | Primer parámetro | ¿Cuándo? |
|---|---|---|---|
| De instancia | ninguno | `self` | Necesita leer o modificar el estado de **un objeto** |
| De clase | `@classmethod` | `cls` | Trabaja con **la clase** (constructores alternativos, contadores...) |
| Estático | `@staticmethod` | ninguno | Función auxiliar relacionada, pero **no usa ni objeto ni clase** |

In [3]:
class Temperatura:
    unidad = "C"

    def __init__(self, grados):
        self.grados = grados

    def describir(self):  # instancia: usa self
        return f"{self.grados}°{self.unidad}"

    @classmethod
    def cambiar_unidad(cls, nueva):  # clase: usa cls
        cls.unidad = nueva

    @staticmethod
    def es_valida(grados):  # estático: ni self ni cls
        return grados >= -273.15


t = Temperatura(21)
print(t.describir())
print(Temperatura.es_valida(-300))
Temperatura.cambiar_unidad("K")
print(t.describir())

21°C
False
21°K


---

## 4. Dataclasses

`@dataclass` es un **decorador** que genera automáticamente, a partir de los atributos
anotados, el `__init__`, la representación al hacer `print` (`__repr__`) y la comparación
con `==` por valor (`__eq__`). Menos código repetitivo, mismas garantías.

- Valores por defecto: `x: int = 0`
- Mutables por defecto: **nunca** `lista: list = []`; usa `field(default_factory=list)`
- Inmutable: `@dataclass(frozen=True)`

In [4]:
from dataclasses import dataclass, field


@dataclass
class Producto:
    nombre: str
    precio: float
    etiquetas: list = field(default_factory=list)  # ✅ lista nueva por instancia


p1 = Producto("Teclado", 49.9)
p2 = Producto("Teclado", 49.9)

print(p1)  # __repr__ gratis
print(p1 == p2)  # __eq__ gratis: compara por valor


@dataclass(frozen=True)
class Punto:
    x: float
    y: float


p = Punto(1, 2)
try:
    p.x = 10
except Exception as e:
    print(type(e).__name__, "->", e)

Producto(nombre='Teclado', precio=49.9, etiquetas=[])
True
FrozenInstanceError -> cannot assign to field 'x'


---

## 5. Herencia y `super()`

- Herencia = relación **"es un"**: `class Perro(Animal)`. La subclase **reutiliza** y puede **ampliar** o **sobrescribir**.
- `super()` llama a la implementación del padre para no repetir código (típico en `__init__`).

In [5]:
class Animal:
    def __init__(self, nombre):
        self.nombre = nombre

    def hablar(self):
        return "..."

    def presentarse(self):
        return f"{self.nombre} dice: {self.hablar()}"


class Perro(Animal):
    def __init__(self, nombre, raza):
        super().__init__(nombre)  # reutiliza el __init__ del padre
        self.raza = raza

    def hablar(self):  # sobrescribe
        return "¡Guau!"


class Gato(Animal):
    def hablar(self):
        return "Miau"


rex = Perro("Rex", "Labrador")
print(rex.presentarse())  # presentarse() viene de Animal, hablar() de Perro
print(Gato("Misi").presentarse())
print(isinstance(rex, Animal))

Rex dice: ¡Guau!
Misi dice: Miau
True


---

## 6. Composición

Composición = relación **"tiene un"**: un objeto guarda otros objetos como atributos y
delega en ellos. Suele ser **más flexible que heredar**: puedes cambiar las piezas sin
tocar la jerarquía.

> 📌 Pregúntate: ¿un `Coche` *es un* `Motor` o *tiene un* `Motor`? Si es "tiene", composición.

In [6]:
class Motor:
    def __init__(self, caballos):
        self.caballos = caballos

    def arrancar(self):
        return f"Motor de {self.caballos} CV arrancado"


class Coche:
    def __init__(self, marca, motor):
        self.marca = marca
        self.motor = motor  # Coche TIENE UN Motor

    def arrancar(self):
        return f"{self.marca}: {self.motor.arrancar()}"  # delega


coche = Coche("Seat", Motor(110))
print(coche.arrancar())

coche.motor = Motor(300)  # cambiar la pieza sin tocar la clase
print(coche.arrancar())

Seat: Motor de 110 CV arrancado
Seat: Motor de 300 CV arrancado


---

## 7. Clases abstractas (ABC)

Una clase abstracta define un **contrato**: qué métodos *deben* implementar las subclases.

- Hereda de `ABC` y marca los métodos con `@abstractmethod`.
- No se puede instanciar la clase abstracta ni una subclase que no implemente todo.
- Beneficio: **fallar pronto** (al crear el objeto), no en mitad de la ejecución.

In [7]:
from abc import ABC, abstractmethod


class Figura(ABC):
    @abstractmethod
    def area(self):
        ...


class Cuadrado(Figura):
    def __init__(self, lado):
        self.lado = lado

    def area(self):
        return self.lado**2


class Incompleta(Figura):
    pass  # olvida implementar area()


print(Cuadrado(3).area())

for clase in (Figura, Incompleta):
    try:
        clase()
    except TypeError as e:
        print("TypeError ->", e)

9
TypeError -> Can't instantiate abstract class Figura without an implementation for abstract method 'area'
TypeError -> Can't instantiate abstract class Incompleta without an implementation for abstract method 'area'


---

## 8. Chuleta final

| Concepto | Recuerda |
|---|---|
| Paradigma POO | Datos + comportamiento juntos en objetos que colaboran, frente a instrucciones sobre datos sueltos |
| Clase / objeto | Molde / pieza. `Clase()` crea una instancia |
| `__init__` + `self` | Inicializa el estado; `self` es el objeto actual |
| Atributo de instancia | `self.x` en `__init__`. Propio de cada objeto |
| Atributo de clase | Definido en el cuerpo de la clase. Compartido por todas las instancias |
| Método de instancia | Recibe `self`. Lee o modifica el estado de un objeto |
| `@classmethod` | Recibe `cls`. Trabaja con la clase |
| `@staticmethod` | Función auxiliar sin `self` ni `cls` |
| `@dataclass` | `__init__`, `__repr__`, `__eq__` gratis. `field(default_factory=list)`, `frozen=True` |
| Herencia | "Es un". `super()` para ampliar sin repetir |
| Composición | "Tiene un". Más flexible: delega en piezas intercambiables |
| ABC | Contrato con `@abstractmethod`. Falla al instanciar, no a mitad de ejecución |